# Smart Parking Dashboard
Ejecuta las celdas en orden. La última celda abre el dashboard en el navegador.

In [ ]:
# !pip install dash plotly pandas numpy

In [ ]:
from dash import Dash, dcc, html, Input, Output
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import os

print('Librerías cargadas')

In [ ]:
# ── RUTA AL CSV ───────────────────────────────────────────────────────────────
CSV_PATH = r"G:\Mi unidad\2 CURSO\Computer Science\trabajocomputer\smart_parking\parking_work\parking_data.csv"

TOTAL_SPACES = 10
GAS_LIMIT    = 450

In [ ]:
# ── CARGAR CSV (datos reales) ─────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()
df["Time"] = pd.to_datetime(df["Time"], format="%H:%M:%S", errors="coerce")
df = df.dropna(subset=["Time"]).reset_index(drop=True)
print(f"CSV cargado: {len(df)} lecturas")
df.head()

In [ ]:
# ── DATOS INVENTADOS POR DÍA DE LA SEMANA (Tab 1) ────────────────────────────
dias = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]

# Coches medios por día
coches_dia = [7, 8, 6, 9, 10, 4, 2]

# Entradas y salidas por día
entradas_dia = [14, 16, 12, 18, 20, 8, 4]
salidas_dia  = [13, 15, 12, 17, 19, 8, 4]

# Calidad del aire media por día
aire_dia = [380, 420, 360, 460, 490, 300, 250]

# Porcentaje de tiempo con barrera bloqueada por día
bloqueado_dia = [10, 15, 8, 22, 30, 5, 2]

print("Datos semanales listos")

In [ ]:
# ── MÉTRICAS DEL CSV (Tab 2) ──────────────────────────────────────────────────
last     = df.iloc[-1]
cars_now = int(last["Cars_inside"])
free_now = int(last["Free_spaces"])
air_now  = int(last["Air_value"])
gate_now = str(last["Gate_status"]).strip()

diff          = df["Cars_inside"].diff().fillna(0)
total_entries = int((diff > 0).sum())
total_exits   = int((diff < 0).sum())

peak_idx  = df["Cars_inside"].idxmax()
peak_time = df.loc[peak_idx, "Time"].strftime("%H:%M:%S")
peak_cars = int(df.loc[peak_idx, "Cars_inside"])
avg_occ   = round(df["Cars_inside"].mean(), 1)
avg_air   = round(df["Air_value"].mean(), 1)
blocked_pct = round(100 * (df["Gate_status"].str.strip() == "BLOCKED").sum() / len(df), 1)

print(f"Coches ahora: {cars_now} | Libres: {free_now} | Barrera: {gate_now}")

In [ ]:
# ── GRÁFICOS TAB 1 ────────────────────────────────────────────────────────────

# Barras: coches medios por día
fig_coches = go.Figure(go.Bar(
    x=dias, y=coches_dia,
    marker_color="steelblue",
    text=coches_dia, textposition="outside",
))
fig_coches.update_layout(
    title="Coches medios por día de la semana",
    yaxis=dict(title="Nº de coches", range=[0, 12]),
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(size=13),
    margin=dict(t=50, b=40), height=320,
)

# Barras agrupadas: entradas vs salidas
fig_flujo = go.Figure([
    go.Bar(name="Entradas", x=dias, y=entradas_dia, marker_color="seagreen"),
    go.Bar(name="Salidas",  x=dias, y=salidas_dia,  marker_color="tomato"),
])
fig_flujo.update_layout(
    title="Entradas y salidas por día",
    barmode="group",
    yaxis=dict(title="Vehículos"),
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(size=13),
    margin=dict(t=50, b=40), height=320,
)

# Línea: calidad del aire por día
fig_aire_dia = go.Figure()
fig_aire_dia.add_hline(y=GAS_LIMIT, line_dash="dash", line_color="red",
                       annotation_text="Límite peligroso",
                       annotation_font_color="red")
fig_aire_dia.add_trace(go.Scatter(
    x=dias, y=aire_dia,
    mode="lines+markers+text",
    text=aire_dia, textposition="top center",
    line=dict(color="darkorange", width=2),
    marker=dict(size=8),
))
fig_aire_dia.update_layout(
    title="Calidad del aire media por día",
    yaxis=dict(title="Valor sensor gas"),
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(size=13),
    margin=dict(t=50, b=40), height=320,
)

# Barras: % tiempo bloqueado por día
fig_bloqueo = go.Figure(go.Bar(
    x=dias, y=bloqueado_dia,
    marker_color="salmon",
    text=[f"{v}%" for v in bloqueado_dia], textposition="outside",
))
fig_bloqueo.update_layout(
    title="% tiempo con barrera bloqueada por día",
    yaxis=dict(title="% del tiempo", range=[0, 40]),
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(size=13),
    margin=dict(t=50, b=40), height=320,
)

print("Gráficos Tab 1 listos")

In [ ]:
# ── GRÁFICOS TAB 2 ────────────────────────────────────────────────────────────

# Línea: ocupación a lo largo del tiempo (CSV)
fig_occ = go.Figure()
fig_occ.add_trace(go.Scatter(
    x=df["Time"], y=df["Cars_inside"],
    mode="lines", name="Coches dentro",
    line=dict(color="steelblue", width=2),
))
fig_occ.add_trace(go.Scatter(
    x=df["Time"], y=df["Free_spaces"],
    mode="lines", name="Plazas libres",
    line=dict(color="seagreen", width=2, dash="dot"),
))
fig_occ.update_layout(
    title="Ocupación registrada (datos reales del CSV)",
    yaxis=dict(title="Nº de coches", range=[0, TOTAL_SPACES + 1]),
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(size=13),
    margin=dict(t=50, b=40), height=320,
)

# Línea: calidad del aire (CSV)
fig_aire = go.Figure()
fig_aire.add_hline(y=GAS_LIMIT, line_dash="dash", line_color="red",
                   annotation_text="Límite peligroso",
                   annotation_font_color="red")
fig_aire.add_trace(go.Scatter(
    x=df["Time"], y=df["Air_value"],
    mode="lines", name="Sensor gas",
    line=dict(color="darkorange", width=2),
))
fig_aire.update_layout(
    title="Calidad del aire registrada (datos reales del CSV)",
    yaxis=dict(title="Valor sensor gas"),
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(size=13),
    margin=dict(t=50, b=40), height=320,
)

# Pie: estado de la barrera (CSV)
gate_counts = df["Gate_status"].str.strip().value_counts()
fig_gate = go.Figure(go.Pie(
    labels=gate_counts.index,
    values=gate_counts.values,
    hole=0.4,
    marker=dict(colors=["seagreen" if l == "ALLOW" else "tomato"
                         for l in gate_counts.index]),
))
fig_gate.update_layout(
    title="Estado de la barrera (ALLOW vs BLOCKED)",
    paper_bgcolor="white",
    font=dict(size=13),
    margin=dict(t=50, b=20), height=320,
)

# Histograma: distribución de ocupación (CSV)
fig_hist = go.Figure(go.Histogram(
    x=df["Cars_inside"].values,
    nbinsx=11,
    marker_color="steelblue",
))
fig_hist.update_layout(
    title="Distribución de la ocupación",
    xaxis=dict(title="Coches dentro", tickvals=list(range(0, TOTAL_SPACES + 1))),
    yaxis=dict(title="Nº de lecturas"),
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(size=13),
    margin=dict(t=50, b=40), height=320,
)

print("Gráficos Tab 2 listos")

In [ ]:
# ── LAYOUT Y CALLBACKS ────────────────────────────────────────────────────────
app = Dash(__name__)

TAB = {"padding": "10px 20px", "fontFamily": "Arial, sans-serif"}
TAB_SEL = {**TAB, "borderTop": "3px solid steelblue", "fontWeight": "bold"}

def kpi(titulo, valor, subtitulo):
    return html.Div([
        html.Div(titulo,    style={"fontSize": "12px", "color": "gray"}),
        html.Div(valor,     style={"fontSize": "32px", "fontWeight": "bold"}),
        html.Div(subtitulo, style={"fontSize": "12px", "color": "gray"}),
    ], style={
        "border": "1px solid #ddd", "borderRadius": "8px",
        "padding": "16px 20px", "flex": "1", "minWidth": "130px",
        "textAlign": "center", "background": "#f9f9f9",
    })

app.layout = html.Div(style={"fontFamily": "Arial, sans-serif",
                              "maxWidth": "1100px", "margin": "0 auto",
                              "padding": "20px"}, children=[

    html.H2("🅿 Smart Parking Dashboard",
            style={"textAlign": "center", "marginBottom": "4px"}),
    html.P("Sistema de monitoreo con Arduino",
           style={"textAlign": "center", "color": "gray", "marginBottom": "24px"}),

    dcc.Tabs(id="tabs", value="tab1", children=[
        dcc.Tab(label="📅  Estadísticas semanales",  value="tab1",
                style=TAB, selected_style=TAB_SEL),
        dcc.Tab(label="📊  Análisis de datos reales", value="tab2",
                style=TAB, selected_style=TAB_SEL),
    ]),

    html.Div(id="contenido", style={"marginTop": "24px"}),
])

@app.callback(Output("contenido", "children"), Input("tabs", "value"))
def render(tab):

    # ── TAB 1: SEMANA ─────────────────────────────────────────────────────────
    if tab == "tab1":
        return html.Div([
            html.P("Datos estimados por día de la semana para un parking de 10 plazas.",
                   style={"color": "gray", "marginBottom": "16px"}),

            # KPIs semana
            html.Div(style={"display": "flex", "gap": "12px",
                            "flexWrap": "wrap", "marginBottom": "24px"}, children=[
                kpi("Día más concurrido", "Viernes", f"{max(coches_dia)} coches de media"),
                kpi("Día más tranquilo",  "Domingo", f"{min(coches_dia)} coches de media"),
                kpi("Total entradas/semana", str(sum(entradas_dia)), "vehículos"),
                kpi("Peor calidad de aire",  "Viernes", f"valor {max(aire_dia)}"),
            ]),

            # Gráficos en grid 2x2
            html.Div(style={"display": "grid",
                            "gridTemplateColumns": "1fr 1fr",
                            "gap": "20px"}, children=[
                dcc.Graph(figure=fig_coches,  config={"displayModeBar": False}),
                dcc.Graph(figure=fig_flujo,   config={"displayModeBar": False}),
                dcc.Graph(figure=fig_aire_dia,config={"displayModeBar": False}),
                dcc.Graph(figure=fig_bloqueo, config={"displayModeBar": False}),
            ]),
        ])

    # ── TAB 2: DATOS REALES ───────────────────────────────────────────────────
    else:
        return html.Div([
            html.P("Datos reales recogidos por el Arduino y guardados en el CSV.",
                   style={"color": "gray", "marginBottom": "16px"}),

            # KPIs CSV
            html.Div(style={"display": "flex", "gap": "12px",
                            "flexWrap": "wrap", "marginBottom": "24px"}, children=[
                kpi("Coches ahora",       str(cars_now),     f"de {TOTAL_SPACES} plazas"),
                kpi("Plazas libres",      str(free_now),     "disponibles"),
                kpi("Entradas totales",   str(total_entries),"registradas"),
                kpi("Salidas totales",    str(total_exits),  "registradas"),
                kpi("Hora punta",         peak_time,         f"{peak_cars} coches"),
                kpi("Ocupación media",    str(avg_occ),      "coches de media"),
                kpi("Aire medio",         str(avg_air),      "valor sensor"),
                kpi("Tiempo bloqueado",   f"{blocked_pct}%", "barrera cerrada"),
            ]),

            # Gráficos en grid 2x2
            html.Div(style={"display": "grid",
                            "gridTemplateColumns": "1fr 1fr",
                            "gap": "20px"}, children=[
                dcc.Graph(figure=fig_occ,  config={"displayModeBar": False}),
                dcc.Graph(figure=fig_aire, config={"displayModeBar": False}),
                dcc.Graph(figure=fig_gate, config={"displayModeBar": False}),
                dcc.Graph(figure=fig_hist, config={"displayModeBar": False}),
            ]),
        ])

print("App lista")

In [ ]:
# ── LANZAR ────────────────────────────────────────────────────────────────────
# Se abre en el navegador como link externo
app.run(jupyter_mode="external", port=8050, debug=False)